# Turbulent box tutorial

## Instalation

The tools we use are RAMSES and Osyris. More information on the codes here:
- https://github.com/nvaytet/osyris
- https://github.com/ramses-organisation/ramses

If you have followed the [general setup tutorial](https://ramses-tutorials.readthedocs.io/en/latest/Setup/general_requirements.html), osyris should already be installed. Otherwise, simply do

In [ ]:
%pip install osyris

You can also install it by cloning directly the source. In this case, please follow the instruction on the GitHub webpage (https://osyris.readthedocs.io/en/latest/installation.html).

To download RAMSES, do in your terminal

In [ ]:
%%bash
git clone https://github.com/ramses-organisation/ramses.git

To compile ramses, you need at least a fortran compiler, either fortran (GNU compiler) or ifort (Intel compiler). You can get more information on the GNU compiler here:
https://gcc.gnu.org/fortran/
https://fortran-lang.org/learn/os_setup/install_gfortran

To run the turbulence driving simulations, you also need the FFTW library. Please follow what is explained in the file ramses/turb/README.

## Run RAMSES

Once you downloaded RAMSES, go in the main directory (`cd ramses`)

### You can first run the test suite:

For each test, there is first a compilation (using the config.txt file of each test), then the code runs (using the namelist files .nml) and finally results are plotted using the `plot-test-name.py` routines.

There are various options for the test suite:    
```bash
#      - Run the suite in parallel (on 4 cpus):
              ./run_test_suite.sh -p 4
#      - Do not delete results data:
             ./run_test_suite.sh -d
#      - Run in verbose mode:
              ./run_test_suite.sh -v
#      - Select test number (for tests 3 to 5, and 10):
              ./run_test_suite.sh -t 3-5,10
#      - Run all tests in mhd directory:
              ./run_test_suite.sh -t mhd
#      - Run quick test suite:
              ./run_test_suite.sh -q
```
The option `-d` is quite useful if you want to work more on the data than what is proposed in the test suite. I would also recommend to use the `-y` option to use python for visualisation.

The turbulence driving test is the test number [16]. You will find in the directory `tests/turb/driving`  the routine plot-driving.py that can be an alternative to the use of osyris. In particular, you can produce column density maps that could be applied to the tools you learned last week!

Of course, you can also play with all the other tests! Note that you can find information on the namelist parameters you can play with here: https://ramses-organisation.readthedocs.io/en/latest/wiki/Runtime_Parameters.html

In [ ]:
%%bash
cd ramses/tests
./run_test_suite.sh -t turb -p 4 

This will run the test suite for the turbulence test case with 4 MPI processes.

### You can then compile directly ramses

Go to the bin directory and update the Makefile: NDIM=3 & USE_TURB=1 

_(note that this corresponds to the FLAGS indicated in the file config.txt of the turbulence driving test)._

**Then**, still in the bin directory


In [ ]:
%%bash
cd ramses/bin
make clean
make NDIM=3 USE_TURB=1 MPI=1 

You should have a file ramses3d that has been created (if not, double check that you updated the Makefile correctly).

In [ ]:
%%bash
ls ramses/bin

You can then run ramses. 
If you have MPI installed you can run on 4 processors: `mpirun -np 4 ramses3d driving.nml`

In [ ]:
%%bash 
cp ramses/bin/ramses3d ramses/tests/turb/driving-static/
cd ramses/tests/turb/driving-static/
./ramses3d driving.nml 

You can now play with the namelist parameters. You can change the resolution (`levelmin=5`, `levelmax=5` for a 32^3 resolution simulation).
You can add more outputs using the parameter foutput in &OUTPUT_PARAMS


```config
&OUTPUT_PARAMS
noutput=2
tout=0.0,0.5
foutput=25 ! Output every 25 steps
/
```



Enjoy!

## Use Osyris to visualise the data

You can check the documentation of osyris [here](https://osyris.readthedocs.io/en/stable/index.html).

### Plot a density slice


Modify the code below to 
- Add an extra layer with the velocity as arrows
- Change the spatial units to pc (you can use the parameter `dx`)
- Change the density unit to solar masses to pc^2
- Change the colormap to inferno

The final output should look like that:

![expected_slice](slice.png)

In [ ]:
import osyris
import numpy as np
import matplotlib.pyplot as plt

data = osyris.RamsesDataset(2, path="ramses/tests/turb/driving-static/").load() # load output number 20. 
center = osyris.Vector(x=2, y=2, z=2, unit="pc") # Choose the center of the box

osyris.map(
    data["mesh"].layer("density"),  # layer 1 : density
    norm="log", origin=center, direction="z")

---
## A numerical experiment: decaying vs driven turbulence

So far we ran a single driven case. Let's turn it into a small **numerical physics
experiment** and actually *measure* how the turbulence behaves. We compare three
setups, all in the same $64^3$ periodic box, based on the turbulence tests in
`ramses/tests/turb/`:

| run | driving | what we expect |
|---|---|---|
| **`decay`** | none &mdash; start from a turbulent field | turbulence **decays** freely |
| **`driven_static`** | continuous, one *frozen* forcing pattern | rises to a **steady state** |
| **`driven_evolving`** | continuous, *time-evolving* (OU) forcing | steady state, time-varying pattern |

The key switches live in `&TURB_PARAMS`: `driven_turb` (drive or not),
`turb_evolving` (frozen vs time-correlated forcing) and `initial_turb` (seed an
initial turbulent field). We also write frequent snapshots (`foutput`) so we can
follow the statistics **as a function of time**. The three namelists `decay.nml`,
`driven_static.nml` and `driven_evolving.nml` are provided next to this notebook.

### Run the three experiments
This runs the three cases into `runs/<case>/`. Each is a cheap $64^3$ run (a few
minutes). If you do **not** have MPI, drop the `mpirun -np 4` and call
`../../ramses/bin/ramses3d` directly.

In [ ]:
%%bash
for case in decay driven_static driven_evolving; do
    rm -rf runs/$case && mkdir -p runs/$case && cp $case.nml runs/$case/
    echo "=== running $case ==="
    ( cd runs/$case && mpirun -np 4 ../../ramses/bin/ramses3d $case.nml > run.log 2>&1 )
    echo "   $(ls -d runs/$case/output_* 2>/dev/null | wc -l) snapshots written"
done

### Visualise the velocity field (density slice + streamlines)
This is the "velocity as arrows" exercise from above, done with streamlines: a
density slice of the **driven, static** run at steady state, with the velocity
field overlaid. You should see turbulent eddies, dense filaments and voids.

*Expected result:*

![velocity streamlines](turb_velocity.png)


In [ ]:
import osyris
import numpy as np
import glob, os
import matplotlib.pyplot as plt

def last_output(path):
    "Index of the last snapshot in a run directory."
    return max(int(os.path.basename(p).split('_')[1]) for p in glob.glob(path + "/output_*"))

run = "runs/driven_static"
data = osyris.RamsesDataset(last_output(run), path=run).load()

osyris.map(
    data["mesh"].layer("density", norm="log"),           # background: density
    data["mesh"].layer("velocity", mode="stream", color="w"),  # overlay: velocity streamlines
    direction="z",
)

### Follow the turbulence statistics in time
We now quantify the three runs. For each snapshot we measure:

- **RMS Mach number** $\mathcal{M} = v_\mathrm{rms}/c_s$, with
  $v_\mathrm{rms}=\langle v^2\rangle^{1/2}$ and the (isothermal) sound speed
  $c_s=\sqrt{P/\rho}$ &mdash; how supersonic the flow is;
- **density dispersion** $\sigma_s = \mathrm{std}(s)$ with $s=\ln(\rho/\langle\rho\rangle)$
  &mdash; how much density structure has formed;
- **kinetic energy** $E_k = \tfrac12\langle\rho v^2\rangle$.

*Expected result:*

![turbulence statistics vs time](turb_stats.png)


In [ ]:
def turb_stats(run):
    "Return arrays (t, Mach, sigma_s, Ekin) over all snapshots of a run."
    snaps = sorted(int(os.path.basename(p).split('_')[1]) for p in glob.glob(run + "/output_*"))
    t, mach, sig, ek = [], [], [], []
    for s in snaps:
        d = osyris.RamsesDataset(s, path=run).load()
        rho = d["mesh"]["density"].values
        v   = d["mesh"]["velocity"]
        v2  = v.x.values**2 + v.y.values**2 + v.z.values**2
        cs  = np.sqrt(d["mesh"]["pressure"].values / rho)          # isothermal sound speed
        t.append(float(d.meta["time"].magnitude) / d.meta["unit_t"])  # time in code units
        mach.append(np.sqrt(np.mean(v2)) / np.mean(cs))
        sig.append(np.std(np.log(rho / np.mean(rho))))
        ek.append(0.5 * np.mean(rho * v2))
    return map(np.array, (t, mach, sig, ek))

runs = {"decay": "C0", "driven_static": "C1", "driven_evolving": "C2"}
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for name, col in runs.items():
    t, mach, sig, ek = turb_stats("runs/" + name)
    ax[0].plot(t, mach, col, marker=".", ms=3, label=name)
    ax[1].plot(t, sig,  col, marker=".", ms=3)
    ax[2].plot(t, ek / ek[ek > 0][0], col, marker=".", ms=3)
ax[0].set(xlabel="t [code]", ylabel=r"$\mathcal{M}$", title="RMS Mach number"); ax[0].legend()
ax[1].set(xlabel="t [code]", ylabel=r"$\sigma_s$", title="density dispersion")
ax[2].set(xlabel="t [code]", ylabel=r"$E_k / E_{k,0}$", title="kinetic energy", yscale="log")
fig.tight_layout()

**What you should see.**
- **`decay`**: a brief settling transient, then $\mathcal{M}$ and $E_k$ **decay**
  monotonically &mdash; with no energy input the turbulence dies away.
- **`driven_static`** and **`driven_evolving`**: $\mathcal{M}$, $\sigma_s$ and $E_k$
  **rise and saturate** &mdash; the forcing injects energy until it balances numerical/shock
  dissipation, reaching a statistical **steady state**. The evolving forcing saturates a
  little lower and noisier than the frozen one.

(These test parameters drive *very* supersonic turbulence, $\mathcal{M}\sim15$&ndash;20.
Lower `turb_rms` for a gentler, more ISM-like $\mathcal{M}\sim5$.)

### The density PDF
A hallmark of supersonic isothermal turbulence is a **log-normal density PDF**: the
logarithmic density $s=\ln(\rho/\langle\rho\rangle)$ is (nearly) Gaussian, with a width set by
the Mach number, $\sigma_s^2 \approx \ln(1+b^2\mathcal{M}^2)$, where $b$ depends on the mix of
solenoidal vs compressive forcing ($b\simeq1/3$ solenoidal, $b\simeq1$ compressive).

*Expected result:*

![density PDF](turb_pdf.png)


In [ ]:
run = "runs/driven_static"
d = osyris.RamsesDataset(last_output(run), path=run).load()
rho = d["mesh"]["density"].values
s = np.log(rho / np.mean(rho))
sig = np.std(s)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(s, bins=60, density=True, alpha=0.6, label="simulation")
x = np.linspace(s.min(), s.max(), 200)
gauss = np.exp(-(x + sig**2/2)**2 / (2*sig**2)) / np.sqrt(2*np.pi*sig**2)  # log-normal: mean = -sig^2/2
ax.plot(x, gauss, "r-", lw=2, label=r"log-normal, $\sigma_s=%.2f$" % sig)
ax.set(xlabel=r"$s=\ln(\rho/\langle\rho\rangle)$", ylabel="PDF",
       title="Density PDF, driven turbulence (steady state)")
ax.legend()

### Solenoidal vs compressive forcing
The *type* of stirring matters as much as its strength. We re-run the driven case with
purely **solenoidal** forcing (`comp_frac=0.0`, divergence-free "stirring") and purely
**compressive** forcing (`comp_frac=1.0`, curl-free "squeezing"), everything else fixed.

In [ ]:
%%bash
for cf in 0.0 1.0; do
    tag=$([ "$cf" = "0.0" ] && echo comp_sol || echo comp_comp)
    rm -rf runs/$tag && mkdir -p runs/$tag
    sed "s/comp_frac=0.5/comp_frac=$cf/" driven_static.nml > runs/$tag/$tag.nml
    ( cd runs/$tag && mpirun -np 4 ../../ramses/bin/ramses3d $tag.nml > run.log 2>&1 )
    echo "$tag done ($(ls -d runs/$tag/output_* | wc -l) snapshots)"
done

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for tag, lab, col in [("comp_sol",  "solenoidal (comp_frac=0)",  "C0"),
                      ("comp_comp", "compressive (comp_frac=1)", "C3")]:
    d   = osyris.RamsesDataset(last_output("runs/" + tag), path="runs/" + tag).load()
    rho = d["mesh"]["density"].values
    s   = np.log(rho / np.mean(rho)); sig = np.std(s)
    v   = d["mesh"]["velocity"]; v2 = v.x.values**2 + v.y.values**2 + v.z.values**2
    M   = np.sqrt(np.mean(v2)) / np.mean(np.sqrt(d["mesh"]["pressure"].values / rho))
    ax.hist(s, bins=60, density=True, alpha=0.5, color=col,
            label=r"%s: $\sigma_s$=%.2f, M=%.0f" % (lab, sig, M))
ax.set(xlabel=r"$s=\ln(\rho/\langle\rho\rangle)$", ylabel="PDF",
       title="Solenoidal vs compressive forcing (same turb_rms)")
ax.legend()

**Compressive forcing produces a much broader density PDF** than solenoidal forcing
*at the same (even slightly lower) Mach number* &mdash; it piles gas into dense sheets and
sweeps out voids far more effectively. In the $\sigma_s^2\approx\ln(1+b^2\mathcal{M}^2)$
relation this is the difference between $b\simeq1/3$ (solenoidal) and $b\simeq1$ (compressive).
This forcing parameter is one of the key knobs in star-formation turbulence studies.

*Expected result:*

![solenoidal vs compressive](turb_comp_sol.png)

### Make a movie of the turbulence developing
As in the dense-core-collapse tutorial, we can turn the snapshots into a movie. Here we take
mid-plane density slices of the driven run across time and stitch them with `ffmpeg`.

In [ ]:
import os, subprocess
from matplotlib.colors import LogNorm

def density_slice(d):
    "Mid-plane (z) density image from a uniform-grid snapshot."
    rho = d["mesh"]["density"].values
    N   = int(round(len(rho) ** (1/3)))
    p   = d["mesh"]["position"]; x, y, z = p.x.values, p.y.values, p.z.values
    zc  = np.unique(z); zmid = zc[len(zc)//2]
    m   = np.abs(z - zmid) < (zc[1]-zc[0]) / 2
    xi  = np.round((x[m]-x[m].min()) / ((x[m].max()-x[m].min())/(N-1))).astype(int)
    yi  = np.round((y[m]-y[m].min()) / ((y[m].max()-y[m].min())/(N-1))).astype(int)
    img = np.full((N, N), np.nan); img[yi, xi] = rho[m]
    return img

run = "runs/driven_static"
snaps = sorted(int(os.path.basename(p).split('_')[1]) for p in glob.glob(run + "/output_*"))
os.makedirs("turb_frames", exist_ok=True)
for n, s in enumerate(snaps):
    d = osyris.RamsesDataset(s, path=run).load()
    fig, a = plt.subplots(figsize=(5, 5))
    a.imshow(density_slice(d), origin="lower", norm=LogNorm(3e-23, 3e-19),
             cmap="inferno", interpolation="nearest")
    a.set_axis_off()
    fig.savefig("turb_frames/f_%04d.png" % n, dpi=90, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

subprocess.call("ffmpeg -y -framerate 12 -i turb_frames/f_%04d.png "
                "-pix_fmt yuv420p -vf scale=512:512 turb_movie.mp4", shell=True)
print("wrote turb_movie.mp4")

A few frames of that movie (gas is near-uniform at first, then develops the familiar
network of shocks, dense filaments and voids as the forcing spins up the turbulence):

![density slices over time](turb_movie_montage.png)

### Going further
- **Forcing spectrum.** Try `forcing_power_spectrum='parabolic'` (see `tests/turb/driving-parabolic`)
  or `'konstandin'`, and compare the density structure.
- **Resolution.** Raise `levelmin=levelmax` to 7 ($128^3$) and watch the small-scale structure sharpen.
- **Power spectrum.** Compute the velocity power spectrum and look for the expected
  slope (Kolmogorov $-5/3$ in the subsonic limit, closer to Burgers $-2$ when strongly supersonic).
